# Fine-tune Qwen3.5-4B with LoRA on the Harmony chest X-ray QA dataset

This Colab notebook fine-tunes [`Qwen/Qwen3.5-4B`](https://huggingface.co/Qwen/Qwen3.5-4B) as a **vision-language** model on the Harmony QA data created by `qwen35_harmony_qa_dataset.ipynb`.

The uploaded Harmony ZIP contains the JSONL labels, but not the radiographs. The notebook therefore uses two inputs:

1. Upload `harmony_qa_dataset.zip` when prompted.
2. Upload a Kaggle API token; the notebook downloads `raddar/chest-xrays-indiana-university` and resolves each JSONL image reference by filename.

LoRA is applied only to the language backbone. The pretrained vision encoder remains frozen. Loss is computed only on the assistant response, never on the system prompt, question, or image tokens. By default both the Harmony `analysis` and `final` channels are supervised; set `TRAIN_REASONING = False` to train only final answers.

> **Runtime:** Use a Colab L4 or A100 GPU. The Kaggle archive is large (approximately 13 GB), so ensure the runtime has enough disk space. This is research/teaching code, not a clinically validated system.

## 1. Install dependencies

Run this cell once, then restart the Colab session before continuing. Qwen3.5 requires a recent Transformers release. This notebook does not use TorchAO quantization, so the install cell removes Colab's optional `torchao` package; an older preinstalled TorchAO can otherwise make PEFT fail while attaching LoRA.

In [ ]:
import subprocess
import sys

PACKAGES = [
    "transformers>=4.57.0",
    "accelerate>=1.2.0",
    "peft>=0.15.0",
    "huggingface_hub>=0.27.0",
    "kaggle>=1.7.0",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-U", *PACKAGES],
    check=True,
)
# PEFT probes torchao whenever it is installed, even though this notebook does not
# use TorchAO quantization. Some Colab images include an old, incompatible build.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)
print("Installation complete and optional torchao removed. Restart the session, then continue with Section 2.")

## 2. Imports and GPU preflight

In [ ]:
import gc
import json
import math
import os
import random
import shutil
import subprocess
import sys
import zipfile
from importlib.metadata import version
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset
from huggingface_hub import notebook_login
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForMultimodalLM, AutoProcessor, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

for package in ["torch", "transformers", "peft", "accelerate", "kaggle"]:
    print(f"{package}: {version(package)}")

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU before continuing.")

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu.total_memory / 2**30:.1f} GB)")
print("BF16 supported:", torch.cuda.is_bf16_supported())

## 3. Upload and unzip the Harmony QA dataset

Upload the ZIP made by the data-generation notebook (for example, `harmony_qa_dataset.zip`). Extraction rejects unsafe archive paths and also accepts a ZIP containing one enclosing directory.

In [ ]:
from google.colab import files

def safe_extract(zip_path, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if os.path.commonpath([str(root), str(target)]) != str(root):
                raise ValueError(f"Unsafe ZIP entry: {member.filename}")
        archive.extractall(destination)

print("Upload harmony_qa_dataset.zip")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError(f"Upload exactly one Harmony dataset ZIP; found {zip_names}")

HARMONY_EXTRACT_DIR = Path("/content/harmony_qa_uploaded")
if HARMONY_EXTRACT_DIR.exists():
    shutil.rmtree(HARMONY_EXTRACT_DIR)
safe_extract(Path("/content") / zip_names[0], HARMONY_EXTRACT_DIR)

train_candidates = list(HARMONY_EXTRACT_DIR.rglob("train.jsonl"))
if len(train_candidates) != 1:
    raise FileNotFoundError(f"Expected exactly one train.jsonl; found {train_candidates}")
HARMONY_DIR = train_candidates[0].parent
TRAIN_PATH = HARMONY_DIR / "train.jsonl"
VAL_PATH = HARMONY_DIR / "val.jsonl"
MANIFEST_PATH = HARMONY_DIR / "manifest.json"
if not VAL_PATH.is_file():
    raise FileNotFoundError(f"Missing {VAL_PATH}")
print("Harmony dataset root:", HARMONY_DIR)
print("Files:", [path.name for path in sorted(HARMONY_DIR.iterdir()) if path.is_file()])

## 4. Load and validate the Harmony records

Validation checks required fields, assistant targets, study-level train/validation separation, and duplicate records.

In [ ]:
def read_jsonl(path):
    records = []
    with Path(path).open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON at {path}:{line_number}") from exc
    return records

def message_text(record, role, channel=None):
    parts = [
        str(message.get("content", ""))
        for message in record.get("messages", [])
        if message.get("role") == role
        and (channel is None or message.get("channel") == channel)
    ]
    return "\n".join(parts).strip()

def validate_records(records, split):
    seen_pairs = set()
    uids = set()
    for index, record in enumerate(records):
        missing = {"uid", "image", "messages"} - set(record)
        if missing:
            raise ValueError(f"{split}[{index}] is missing {sorted(missing)}")
        question = message_text(record, "user")
        final = message_text(record, "assistant", "final")
        if "<image>" not in question or not final:
            raise ValueError(f"{split}[{index}] lacks an image marker or final answer")
        key = (str(record["uid"]), question)
        if key in seen_pairs:
            raise ValueError(f"{split}[{index}] duplicates uid/question {key}")
        seen_pairs.add(key)
        uids.add(str(record["uid"]))
    return uids

train_records = read_jsonl(TRAIN_PATH)
val_records = read_jsonl(VAL_PATH)
if not train_records or not val_records:
    raise ValueError("Both train.jsonl and val.jsonl must be non-empty.")
train_uids = validate_records(train_records, "train")
val_uids = validate_records(val_records, "validation")
if train_uids & val_uids:
    raise ValueError(f"Study leakage between splits: {sorted(train_uids & val_uids)}")

manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.is_file() else {}
print(json.dumps(manifest, indent=2))
print(f"Train: {len(train_records)} QA pairs from {len(train_uids)} studies")
print(f"Validation: {len(val_records)} QA pairs from {len(val_uids)} studies")
print("Example question:", message_text(train_records[0], "user").replace("<image>", "").strip())

## 5. Authenticate with Kaggle and download the IU chest X-ray dataset

From [Kaggle Settings](https://www.kaggle.com/settings), create an API token and upload `kaggle.json` when prompted. The legacy `kaggle_API.txt` token used by the data-generation notebook is also accepted. Keep credentials private.

In [ ]:
print("Upload kaggle.json (or kaggle_API.txt)")
credential_upload = files.upload()
credential_names = list(credential_upload)
if len(credential_names) != 1:
    raise ValueError(f"Upload exactly one Kaggle credential file; found {credential_names}")

credential_name = credential_names[0]
credential_path = Path("/content") / credential_name
KAGGLE_CONFIG_DIR = Path("/root/.kaggle")
KAGGLE_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
if credential_name.lower() == "kaggle.json":
    target = KAGGLE_CONFIG_DIR / "kaggle.json"
elif credential_name.lower().endswith(".txt"):
    target = KAGGLE_CONFIG_DIR / "access_token"
else:
    raise ValueError("Expected kaggle.json or a .txt Kaggle access token.")
shutil.copy2(credential_path, target)
os.chmod(target, 0o600)
print("Installed Kaggle credential as", target)

In [ ]:
KAGGLE_DATASET = "raddar/chest-xrays-indiana-university"
KAGGLE_DOWNLOAD_DIR = Path("/content/kaggle_iu_download")
KAGGLE_EXTRACT_DIR = Path("/content/iu_chest_xray")
KAGGLE_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [sys.executable, "-m", "kaggle", "datasets", "download",
     "-d", KAGGLE_DATASET, "-p", str(KAGGLE_DOWNLOAD_DIR)],
    check=True,
)
downloaded_zips = list(KAGGLE_DOWNLOAD_DIR.glob("*.zip"))
if len(downloaded_zips) != 1:
    raise FileNotFoundError(f"Expected one Kaggle ZIP; found {downloaded_zips}")
if KAGGLE_EXTRACT_DIR.exists():
    shutil.rmtree(KAGGLE_EXTRACT_DIR)
safe_extract(downloaded_zips[0], KAGGLE_EXTRACT_DIR)
print("Kaggle dataset extracted to:", KAGGLE_EXTRACT_DIR)
print("PNG files found:", sum(1 for _ in KAGGLE_EXTRACT_DIR.rglob("*.png")))

## 6. Resolve and verify every radiograph

The Kaggle archive can change its enclosing directory layout. Matching by unique PNG basename avoids hard-coding that layout while still failing loudly on missing or ambiguous images.

In [ ]:
from collections import defaultdict

paths_by_name = defaultdict(list)
for image_path in KAGGLE_EXTRACT_DIR.rglob("*.png"):
    paths_by_name[image_path.name].append(image_path)

def attach_image_paths(records, split):
    missing = []
    ambiguous = []
    for record in records:
        basename = Path(record["image"]).name
        matches = paths_by_name.get(basename, [])
        if not matches:
            missing.append(basename)
        elif len(matches) > 1:
            ambiguous.append((basename, matches))
        else:
            record["_image_path"] = str(matches[0])
    if missing:
        raise FileNotFoundError(f"{split}: {len(missing)} images missing; examples: {missing[:5]}")
    if ambiguous:
        raise ValueError(f"{split}: ambiguous image basenames; first: {ambiguous[0]}")

attach_image_paths(train_records, "train")
attach_image_paths(val_records, "validation")
print(f"Resolved all {len(train_records) + len(val_records)} record image references.")
print("Example:", train_records[0]["_image_path"])

## 7. Hugging Face login and training configuration

The model is public, but logging in avoids anonymous download rate limits. If already authenticated in this runtime, the login widget may be skipped.

In [ ]:
notebook_login()

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-4B"
OUTPUT_DIR = Path("/content/qwen35_harmony_qa_lora")
SEED = 42
MAX_LENGTH = 3072
MAX_IMAGE_TOKENS = 1024
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TRAIN_REASONING = True  # False trains only the Harmony final answer.

QWEN_VISION_PIXEL_FACTOR = 32  # 16-pixel patches followed by 2x2 merging
MIN_IMAGE_TOKENS = 256
MIN_IMAGE_PIXELS = MIN_IMAGE_TOKENS * QWEN_VISION_PIXEL_FACTOR**2
MAX_IMAGE_PIXELS = MAX_IMAGE_TOKENS * QWEN_VISION_PIXEL_FACTOR**2
MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

random.seed(SEED)
torch.manual_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({"dtype": str(MODEL_DTYPE), "train_reasoning": TRAIN_REASONING, "output": str(OUTPUT_DIR)})

## 8. Convert Harmony channels to Qwen conversations

Harmony stores `analysis` and `final` as separate assistant messages. Qwen uses a single assistant turn, so reasoning supervision is represented by a `<think>…</think>` block followed by the final answer. The system and developer messages are combined because Qwen's chat template has one system role.

In [ ]:
def qwen_messages(record, include_assistant=True):
    system = message_text(record, "system")
    developer = message_text(record, "developer")
    question = message_text(record, "user").replace("<image>", "").strip()
    messages = []
    combined_instructions = "\n\n".join(part for part in [system, developer] if part).strip()
    if combined_instructions:
        messages.append({"role": "system", "content": combined_instructions})
    messages.append({
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": question}],
    })
    if include_assistant:
        reasoning = message_text(record, "assistant", "analysis")
        final = message_text(record, "assistant", "final")
        answer = f"<think>\n{reasoning}\n</think>\n{final}" if TRAIN_REASONING and reasoning else final
        messages.append({"role": "assistant", "content": answer})
    return messages

def load_cxr(path):
    with Image.open(path) as image_file:
        image = image_file.convert("RGB")
    area = image.width * image.height
    if area <= MAX_IMAGE_PIXELS:
        return image
    scale = math.sqrt(MAX_IMAGE_PIXELS / area)
    width = max(32, int(image.width * scale) // 32 * 32)
    height = max(32, int(image.height * scale) // 32 * 32)
    return image.resize((width, height), Image.Resampling.LANCZOS)

print(json.dumps(qwen_messages(train_records[0]), indent=2)[:3000])

## 9. Load Qwen3.5-4B and attach LoRA

The target discovery uses full module names from the installed Qwen implementation and excludes the vision tower, embeddings, and output head. This is more robust than assuming a fixed list of projection names.

In [ ]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_IMAGE_PIXELS,
    max_pixels=MAX_IMAGE_PIXELS,
)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

VISION_MARKERS = ("visual", "vision_model", "vision_tower", "vision_encoder")
TEXT_MARKERS = ("language_model", "text_model", "model.layers")
EXCLUDE_MARKERS = ("lm_head", "embed_tokens", "embedding", "mtp")
target_modules = sorted({
    name
    for name, module in model.named_modules()
    if isinstance(module, torch.nn.Linear)
    and any(marker in name.lower() for marker in TEXT_MARKERS)
    and not any(marker in name.lower() for marker in VISION_MARKERS + EXCLUDE_MARKERS)
})
if not target_modules:
    sample_names = [name for name, module in model.named_modules() if isinstance(module, torch.nn.Linear)][:30]
    raise RuntimeError(f"No text LoRA targets found. Example linear layers: {sample_names}")
print(f"LoRA target modules: {len(target_modules)}")
print("First targets:", target_modules[:20])

model = get_peft_model(model, LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=target_modules,
    task_type="CAUSAL_LM",
))
model.print_trainable_parameters()

## 10. Response-only multimodal dataset and collator

Each sample is processed twice to measure the prompt boundary. Labels before that boundary are set to `-100`, so only assistant tokens contribute to the loss. Multimodal truncation is deliberately disabled; the notebook raises an error instead of silently dropping image or answer tokens.

In [ ]:
def render_chat(messages, add_generation_prompt):
    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=False,
    )

class HarmonyImageQADataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = load_cxr(record["_image_path"])
        prompt_text = render_chat(qwen_messages(record, False), True)
        full_text = render_chat(qwen_messages(record, True), False)
        full = processor(text=full_text, images=image, return_tensors="pt", truncation=False)
        prompt = processor(text=prompt_text, images=image, return_tensors="pt", truncation=False)

        full_length = int(full["input_ids"].shape[-1])
        prompt_length = int(prompt["input_ids"].shape[-1])
        if full_length > MAX_LENGTH:
            raise ValueError(
                f"uid={record['uid']} has {full_length} tokens, above MAX_LENGTH={MAX_LENGTH}. "
                "Reduce MAX_IMAGE_TOKENS or increase MAX_LENGTH."
            )
        if prompt_length >= full_length:
            raise ValueError(f"uid={record['uid']} has no assistant target tokens.")

        item = {key: value.squeeze(0) for key, value in full.items()}
        labels = item["input_ids"].clone()
        labels[:prompt_length] = -100
        item["labels"] = labels
        return item

class MultimodalCollator:
    def __init__(self):
        tokenizer = processor.tokenizer
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"
        self.pad_token_id = tokenizer.pad_token_id

    @staticmethod
    def pad(tensor, length, value):
        if tensor.shape[0] == length:
            return tensor
        shape = (length - tensor.shape[0],) + tuple(tensor.shape[1:])
        return torch.cat([tensor, torch.full(shape, value, dtype=tensor.dtype)], dim=0)

    def __call__(self, features):
        batch = {}
        sequence_keys = {"input_ids", "attention_mask", "token_type_ids", "mm_token_type_ids", "labels"}
        max_length = max(feature["input_ids"].shape[0] for feature in features)
        for key in sequence_keys:
            if all(key in feature for feature in features):
                value = -100 if key == "labels" else (self.pad_token_id if key == "input_ids" else 0)
                batch[key] = torch.stack([self.pad(feature[key], max_length, value) for feature in features])

        common_keys = set.intersection(*(set(feature) for feature in features)) - sequence_keys
        for key in sorted(common_keys):
            values = [feature[key] for feature in features]
            if key in {"pixel_values", "pixel_values_videos"}:
                batch[key] = torch.cat(values, dim=0)
            else:
                batch[key] = torch.stack(values, dim=0)
        return batch

train_dataset = HarmonyImageQADataset(train_records)
val_dataset = HarmonyImageQADataset(val_records)
collator = MultimodalCollator()

## 11. Collator and model forward-pass preflight

This catches sequence-length, multimodal batching, and model-input errors before training starts.

In [ ]:
probe_count = min(2, len(train_dataset))
probe = collator([train_dataset[index] for index in range(probe_count)])
print("Batch shapes:", {key: tuple(value.shape) for key, value in probe.items()})

input_device = next(parameter.device for parameter in model.parameters() if parameter.device.type != "meta")
probe_on_device = {
    key: value.to(input_device, dtype=MODEL_DTYPE) if value.is_floating_point() else value.to(input_device)
    for key, value in probe.items()
}
with torch.no_grad():
    probe_loss = model(**probe_on_device).loss.item()
print("Forward-pass loss:", probe_loss)
del probe, probe_on_device
gc.collect()
torch.cuda.empty_cache()

## 12. Train and save the best LoRA adapter

Evaluation and checkpointing run once per epoch. Re-running this cell resumes from the latest checkpoint in the Colab runtime.

In [ ]:
checkpoint_dir = OUTPUT_DIR / "checkpoints"
training_args = TrainingArguments(
    output_dir=str(checkpoint_dir),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=(MODEL_DTYPE == torch.bfloat16),
    fp16=(MODEL_DTYPE == torch.float16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    remove_unused_columns=False,
    prediction_loss_only=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
)
last_checkpoint = get_last_checkpoint(str(checkpoint_dir)) if checkpoint_dir.exists() else None
if last_checkpoint:
    print("Resuming from:", last_checkpoint)
trainer.train(resume_from_checkpoint=last_checkpoint)
metrics = trainer.evaluate()

ADAPTER_DIR = OUTPUT_DIR / "best_adapter"
trainer.save_model(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(OUTPUT_DIR / "run_config.json").write_text(json.dumps({
    "model_id": MODEL_ID,
    "train_reasoning": TRAIN_REASONING,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "max_length": MAX_LENGTH,
    "max_image_tokens": MAX_IMAGE_TOKENS,
    "epochs": NUM_TRAIN_EPOCHS,
    "train_records": len(train_records),
    "validation_records": len(val_records),
}, indent=2))
print("Validation metrics:", metrics)
print("Saved adapter:", ADAPTER_DIR)

## 13. Held-out qualitative generation check

This checks end-to-end image loading and generation on one validation example. It is not a substitute for clinical evaluation.

In [ ]:
@torch.inference_mode()
def generate_answer(record, max_new_tokens=512):
    prompt_text = render_chat(qwen_messages(record, False), True)
    image = load_cxr(record["_image_path"])
    inputs = processor(text=prompt_text, images=image, return_tensors="pt")
    device = next(parameter.device for parameter in model.parameters() if parameter.device.type != "meta")
    inputs = {
        key: value.to(device, dtype=MODEL_DTYPE) if value.is_floating_point() else value.to(device)
        for key, value in inputs.items()
    }
    was_training = model.training
    model.eval()
    output = model.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=max_new_tokens,
        pad_token_id=processor.tokenizer.pad_token_id,
    )
    if was_training:
        model.train()
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()

example = val_records[0]
print("Question:", message_text(example, "user").replace("<image>", "").strip())
print("\nReference final:", message_text(example, "assistant", "final"))
print("\nGenerated:", generate_answer(example))

## 14. Download the adapter

The ZIP contains LoRA weights/configuration, processor files, metrics, and the run configuration. It does **not** contain the Qwen3.5 base-model weights.

In [ ]:
archive_base = Path("/content/qwen35_harmony_qa_lora_adapter")
shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR)
archive_path = archive_base.with_suffix(".zip")
print(f"Created {archive_path} ({archive_path.stat().st_size / 2**20:.1f} MB)")
files.download(str(archive_path))